In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamb/netflix-shows")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'netflix-shows' dataset.
Path to dataset files: /kaggle/input/netflix-shows


In [2]:
import duckdb
import os

# Connect to in-memory DuckDB
con1 = duckdb.connect()

In [4]:
csv_netflix_path = os.path.join(path, 'netflix_titles.csv')
con1.execute(f"CREATE OR REPLACE TABLE netflix AS SELECT * FROM '{csv_netflix_path}'");

In [5]:
q11="""
WITH RatingCounts AS (
    SELECT
        type,
        rating,
        COUNT(*) AS total_count
    FROM netflix
    WHERE rating IS NOT NULL
    GROUP BY type, rating
),
RankedRatings AS (
    SELECT
        type,
        rating,
        total_count,
        RANK() OVER (PARTITION BY type ORDER BY total_count DESC) AS ranking
    FROM RatingCounts
)
SELECT
    type,
    rating AS dominant_rating,
    total_count
FROM RankedRatings
WHERE ranking = 1;
"""
df_result4=con1.execute(q11).df()
display(df_result4)

,type,dominant_rating,total_count
0,Movie,TV-MA,2062
1,TV Show,TV-MA,1145


In [6]:
q12="""
SELECT
    release_year,
    COUNT(show_id) AS total_release,
    ROUND(
        COUNT(show_id)::numeric /
        (SELECT COUNT(show_id) FROM netflix WHERE country = 'India')::numeric * 100,
        2
    ) AS pct_of_total_india_catalog
FROM netflix
WHERE country = 'India'
GROUP BY release_year
ORDER BY pct_of_total_india_catalog DESC
LIMIT 5;
"""
df_result5=con1.execute(q12).df()
display(df_result5)

,release_year,total_release,pct_of_total_india_catalog
0,2017,101,10.39
1,2018,94,9.67
2,2019,87,8.95
3,2020,75,7.72
4,2016,73,7.51


In [9]:
q13="""
SELECT
    TRIM(t2.genre_element) AS genre,
    COUNT(t.show_id) AS total_content
FROM netflix AS t,
     LATERAL UNNEST(STRING_TO_ARRAY(t.listed_in, ',')) AS t2(genre_element)
GROUP BY TRIM(t2.genre_element)
ORDER BY total_content DESC
LIMIT 10;
"""
df_result6=con1.execute(q13).df()
display(df_result6)

,genre,total_content
0,International Movies,2752
1,Dramas,2427
2,Comedies,1674
3,International TV Shows,1351
4,Documentaries,869
5,Action & Adventure,859
6,TV Dramas,763
7,Independent Movies,756
8,Children & Family Movies,641
9,Romantic Movies,616


In [10]:
q14="""
SELECT
    CASE
        WHEN description ILIKE '%kill%' OR description ILIKE '%violence%' THEN 'Sensitive / Mature'
        ELSE 'Family / General'
    END AS content_category,
    COUNT(*) AS total_titles,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM netflix), 2) AS share_pct
FROM netflix
GROUP BY 1;
"""
df_result7=con1.execute(q14).df()
display(df_result7)

,content_category,total_titles,share_pct
0,Family / General,8465,96.12
1,Sensitive / Mature,342,3.88
